# 03 — Analysis & Visualisation

Loads scores, produces bar/box plots, and prints a summary table.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

import config
from src.generation import load_stories

In [ ]:
# ── Load scored stories ───────────────────────────────────────────────────────
rows = []
for sched_name in config.SCHEDULES:
    out_dir = config.results_dir(config.MODEL_NAME, sched_name)
    scores_path = os.path.join(out_dir, "scores.jsonl")
    if not os.path.exists(scores_path):
        print(f"WARNING: {scores_path} not found")
        continue
    for s in load_stories(scores_path):
        rows.append(s)

df = pd.DataFrame(rows)
print(df.shape, df.columns.tolist())
df.head(2)

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
score_cols = [c for c in ["coherence", "coherence_llm", "creativity"] if c in df.columns]
summary = df.groupby("schedule")[score_cols].agg(["mean", "std"]).round(4)
print(summary.to_string())

In [ ]:
# ── Bar chart: mean coherence per schedule ────────────────────────────────────
if "coherence" in df.columns:
    means = df.groupby("schedule")["coherence"].mean().reindex(config.SCHEDULES)
    stds  = df.groupby("schedule")["coherence"].std().reindex(config.SCHEDULES)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
           color="steelblue", alpha=0.8)
    ax.set_title("Mean Log-Likelihood Coherence by Schedule")
    ax.set_xlabel("Schedule")
    ax.set_ylabel("Avg token log-prob")
    plt.tight_layout()
    plt.savefig(os.path.join(config.RESULTS_DIR, "coherence_bar.png"), dpi=150)
    plt.show()

In [ ]:
# ── Box plots: coherence distribution ────────────────────────────────────────
if "coherence" in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    data = [df[df["schedule"] == s]["coherence"].dropna().values for s in config.SCHEDULES]
    ax.boxplot(data, labels=config.SCHEDULES, patch_artist=True)
    ax.set_title("Coherence Distribution by Schedule")
    ax.set_xlabel("Schedule")
    ax.set_ylabel("Avg token log-prob")
    plt.tight_layout()
    plt.savefig(os.path.join(config.RESULTS_DIR, "coherence_box.png"), dpi=150)
    plt.show()

In [ ]:
# ── LLM judge scores (Gemini mode) ────────────────────────────────────────────
if "creativity" in df.columns and df["creativity"].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, title in zip(
        axes,
        ["creativity", "coherence_llm"],
        ["Creativity (Gemini judge)", "Coherence (Gemini judge)"],
    ):
        means = df.groupby("schedule")[col].mean().reindex(config.SCHEDULES)
        stds  = df.groupby("schedule")[col].std().reindex(config.SCHEDULES)
        ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
               color="coral", alpha=0.8)
        ax.set_title(title)
        ax.set_ylim(1, 5)
        ax.set_xlabel("Schedule")
        ax.set_ylabel("Score (1–5)")
    plt.tight_layout()
    plt.savefig(os.path.join(config.RESULTS_DIR, "llm_judge_bars.png"), dpi=150)
    plt.show()

In [ ]:
# ── Diversity (self-BLEU) ─────────────────────────────────────────────────────
div_path = os.path.join(config.RESULTS_DIR, "diversity.json")
if os.path.exists(div_path):
    with open(div_path) as f:
        diversity = json.load(f)

    fig, ax = plt.subplots(figsize=(7, 4))
    scheds = [s for s in config.SCHEDULES if s in diversity]
    ax.bar(scheds, [diversity[s] for s in scheds], color="seagreen", alpha=0.8)
    ax.set_title("Self-BLEU Diversity by Schedule (lower = more diverse)")
    ax.set_xlabel("Schedule")
    ax.set_ylabel("Self-BLEU")
    plt.tight_layout()
    plt.savefig(os.path.join(config.RESULTS_DIR, "diversity_bar.png"), dpi=150)
    plt.show()
    print("Diversity scores:", diversity)